# 06 - Verificar APRX de publicacion

Control de calidad para revisar si las imagenes incluidas en la query de footprints por campo `Name` estan presentes en el APRX actualizado para publicacion.

La validacion no modifica el APRX. Solo abre el proyecto, localiza el mapa y el grupo `Vuelos Drone PAO > Imagenes Drone`, compara nombres esperados contra capas raster existentes y genera CSV de revision.

## 0. Configuracion

La fuente principal de nombres es `05_where_name_footprints_names.csv`, generado desde los resultados de carga. Si ese archivo no existe, se toma la columna `Name` desde `02_load_results.csv`.

In [ ]:
from pathlib import Path
from datetime import datetime
import csv
import re

import arcpy

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT.name.lower() == 'flujo_geosupport_etapas' or not (PROJECT_ROOT / 'flujo_geosupport_etapas').exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError('No se pudo resolver PROJECT_ROOT')
    PROJECT_ROOT = PROJECT_ROOT.parent

FLOW_DIR = PROJECT_ROOT / 'flujo_geosupport_etapas'

QUERY_NAMES_CSV = FLOW_DIR / 'outputs' / 'etapa_03_actualizar_footprints_indice' / '05_where_name_footprints_names.csv'
LOAD_RESULTS_CSV = FLOW_DIR / 'outputs' / 'etapa_02_carga_datastore_mosaico' / '02_load_results.csv'

# APRX oficial usado para publicacion. Si se quiere revisar una copia local, cambiar este path.
APRX_PATH = r"\\amssclgis08.ams.gmams.cl\CL_MLP_PAO\01_Proyectos_ArcGIS\APRX\VISOR TERRITORIAL SIG PAO v7.aprx"
# Ejemplo copia de revision generada por etapa 05:
# APRX_PATH = str(PROJECT_ROOT / 'APRX' / 'VISOR TERRITORIAL SIG PAO v7_verificacion_imagenes_drone.aprx')

MAP_NAME = 'CL MLP PAO 27 Imagenes Aereas PAO Image Server'
PARENT_GROUP_NAME = 'Vuelos Drone PAO'
TARGET_GROUP_NAME = 'Imagenes Drone'

PREFIX = 'CL_MLP_PAO_IF_Ortho_'
RASTER_SUFFIXES = ('.tif', '.tiff')

# True: solo capas raster hijas directas del grupo Imagenes Drone.
# False: permite encontrar capas raster descendientes dentro de subgrupos.
CHECK_DIRECT_CHILDREN_ONLY = True

OUTPUT_DIR = FLOW_DIR / 'outputs' / 'etapa_06_verificar_aprx_publicacion'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('CSV names query:', QUERY_NAMES_CSV)
print('CSV carga etapa 2:', LOAD_RESULTS_CSV)
print('APRX:', APRX_PATH)
print('Mapa:', MAP_NAME)
print('Grupo:', f'{PARENT_GROUP_NAME} > {TARGET_GROUP_NAME}')
print('Salida:', OUTPUT_DIR)

## 1. Leer nombres esperados desde query/resultados de carga

In [ ]:
def read_csv_rows(path):
    with open(path, 'r', encoding='utf-8-sig', newline='') as handle:
        return list(csv.DictReader(handle))


def read_expected_names():
    source = None
    if QUERY_NAMES_CSV.exists():
        rows = read_csv_rows(QUERY_NAMES_CSV)
        source = QUERY_NAMES_CSV
    elif LOAD_RESULTS_CSV.exists():
        rows = read_csv_rows(LOAD_RESULTS_CSV)
        source = LOAD_RESULTS_CSV
    else:
        raise FileNotFoundError(f'No existe {QUERY_NAMES_CSV} ni {LOAD_RESULTS_CSV}')

    names = sorted({str(row.get('Name', '')).strip() for row in rows if str(row.get('Name', '')).strip()})
    return names, source


expected_names, expected_source = read_expected_names()
print('Fuente nombres:', expected_source)
print('Nombres esperados:', len(expected_names))
for name in expected_names[:10]:
    print('-', name)

## 2. Utilidades de comparacion contra capas del APRX

In [ ]:
def layer_long_name(layer):
    try:
        return layer.longName
    except Exception:
        return layer.name


def short_image_name(value):
    name = Path(str(value)).name
    if name.startswith('tmp_'):
        name = name.replace('tmp_', '', 1)
    if name.startswith(PREFIX):
        name = name.replace(PREFIX, '', 1)
    lower = name.lower()
    for suffix in RASTER_SUFFIXES:
        if lower.endswith(suffix):
            name = name[:-len(suffix)]
            break
    return name


def image_keys(value):
    if value is None:
        return set()
    raw = str(value).strip()
    if not raw:
        return set()
    basename = Path(raw).name
    stem = Path(basename).stem
    short = short_image_name(raw)
    keys = {
        raw.lower(),
        basename.lower(),
        stem.lower(),
        short.lower(),
        f'{PREFIX}{short}'.lower(),
        f'{PREFIX}{short}.tif'.lower(),
        f'{PREFIX}{short}.tiff'.lower(),
        f'{short}.tif'.lower(),
        f'{short}.tiff'.lower(),
    }
    return {key for key in keys if key}


def safe_data_source(layer):
    try:
        return layer.dataSource
    except Exception:
        return None


def find_group(map_obj, group_name, parent_group=None):
    groups = [layer for layer in map_obj.listLayers() if layer.isGroupLayer]
    matches = []
    for group in groups:
        if group.name != group_name:
            continue
        if parent_group is not None:
            expected_prefix = layer_long_name(parent_group) + '\\'
            if not layer_long_name(group).startswith(expected_prefix):
                continue
        matches.append(group)
    if not matches:
        available = [layer_long_name(layer) for layer in groups]
        parent_text = f' bajo {layer_long_name(parent_group)}' if parent_group else ''
        raise ValueError(f"No se encontro grupo '{group_name}'{parent_text}. Grupos disponibles: {available}")
    if len(matches) > 1:
        print(f"Advertencia: se encontraron {len(matches)} grupos '{group_name}'. Se usara {layer_long_name(matches[0])}")
    return matches[0]


def is_descendant(layer, group_layer):
    long_name = layer_long_name(layer)
    group_long_name = layer_long_name(group_layer)
    return long_name.startswith(group_long_name + '\\')


def is_direct_child(layer, group_layer):
    long_name = layer_long_name(layer)
    group_long_name = layer_long_name(group_layer)
    prefix = group_long_name + '\\'
    if not long_name.startswith(prefix):
        return False
    relative = long_name[len(prefix):]
    return '\\' not in relative


def group_raster_layers(map_obj, group_layer, direct_only=True):
    layers = []
    for layer in map_obj.listLayers():
        if layer == group_layer or layer.isGroupLayer or not layer.isRasterLayer:
            continue
        if direct_only and is_direct_child(layer, group_layer):
            layers.append(layer)
        elif not direct_only and is_descendant(layer, group_layer):
            layers.append(layer)
    return layers

## 3. Revisar APRX y generar resultados

In [ ]:
run_timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
validation_csv = OUTPUT_DIR / '01_aprx_query_validation.csv'
inventory_csv = OUTPUT_DIR / '02_aprx_group_layer_inventory.csv'
summary_csv = OUTPUT_DIR / '00_summary.csv'

aprx = arcpy.mp.ArcGISProject(APRX_PATH)
try:
    maps = aprx.listMaps(MAP_NAME)
    if not maps:
        raise ValueError(f'No se encontro el mapa: {MAP_NAME}')
    map_obj = maps[0]
    parent_group = find_group(map_obj, PARENT_GROUP_NAME)
    target_group = find_group(map_obj, TARGET_GROUP_NAME, parent_group)

    print('Grupo padre:', layer_long_name(parent_group))
    print('Grupo destino:', layer_long_name(target_group))

    layers = group_raster_layers(map_obj, target_group, direct_only=CHECK_DIRECT_CHILDREN_ONLY)
    inventory_rows = []
    layer_key_index = {}

    for index, layer in enumerate(layers, start=1):
        data_source = safe_data_source(layer)
        keys = image_keys(layer.name)
        keys.update(image_keys(data_source))
        row = {
            'layer_order': index,
            'layer_name': layer.name,
            'layer_short_name': short_image_name(layer.name),
            'layer_long_name': layer_long_name(layer),
            'data_source': data_source,
            'key_count': len(keys),
        }
        inventory_rows.append(row)
        for key in keys:
            layer_key_index.setdefault(key, []).append(row)

    validation_rows = []
    for name in expected_names:
        expected_short = short_image_name(name)
        keys = image_keys(name)
        matches = []
        seen = set()
        for key in keys:
            for match in layer_key_index.get(key, []):
                match_id = (match['layer_order'], match['layer_long_name'])
                if match_id not in seen:
                    matches.append(match)
                    seen.add(match_id)

        validation_rows.append({
            'Name': name,
            'expected_layer_name': expected_short,
            'status': 'found' if matches else 'missing',
            'match_count': len(matches),
            'matched_layer_names': '|'.join(match['layer_name'] for match in matches),
            'matched_layer_long_names': '|'.join(match['layer_long_name'] for match in matches),
            'matched_data_sources': '|'.join(str(match.get('data_source') or '') for match in matches),
        })

    with open(inventory_csv, 'w', encoding='utf-8-sig', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=['layer_order', 'layer_name', 'layer_short_name', 'layer_long_name', 'data_source', 'key_count'])
        writer.writeheader()
        writer.writerows(inventory_rows)

    with open(validation_csv, 'w', encoding='utf-8-sig', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=['Name', 'expected_layer_name', 'status', 'match_count', 'matched_layer_names', 'matched_layer_long_names', 'matched_data_sources'])
        writer.writeheader()
        writer.writerows(validation_rows)

    found_count = sum(1 for row in validation_rows if row['status'] == 'found')
    missing_rows = [row for row in validation_rows if row['status'] == 'missing']
    duplicate_rows = [row for row in validation_rows if row['match_count'] > 1]

    summary_rows = [
        {'metric': 'run_timestamp', 'value': run_timestamp},
        {'metric': 'expected_source', 'value': str(expected_source)},
        {'metric': 'aprx_path', 'value': APRX_PATH},
        {'metric': 'map_name', 'value': MAP_NAME},
        {'metric': 'target_group', 'value': f'{PARENT_GROUP_NAME} > {TARGET_GROUP_NAME}'},
        {'metric': 'direct_children_only', 'value': CHECK_DIRECT_CHILDREN_ONLY},
        {'metric': 'expected_names_count', 'value': len(expected_names)},
        {'metric': 'group_raster_layers_count', 'value': len(layers)},
        {'metric': 'found_count', 'value': found_count},
        {'metric': 'missing_count', 'value': len(missing_rows)},
        {'metric': 'duplicate_match_count', 'value': len(duplicate_rows)},
    ]
    with open(summary_csv, 'w', encoding='utf-8-sig', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=['metric', 'value'])
        writer.writeheader()
        writer.writerows(summary_rows)

    print(f'Esperadas: {len(expected_names)}')
    print(f'Capas raster en grupo: {len(layers)}')
    print(f'Encontradas: {found_count}')
    print(f'Faltantes: {len(missing_rows)}')
    print(f'Matches duplicados: {len(duplicate_rows)}')

    if missing_rows:
        print('Faltantes:')
        for row in missing_rows:
            print('-', row['Name'])

    print('Resumen:', summary_csv)
    print('Validacion:', validation_csv)
    print('Inventario grupo:', inventory_csv)
finally:
    del aprx